# Audio Flamingo 3 (NVIDIA)

In [20]:
import os
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor

In [21]:
!source /etc/network_turbo

设置成功
注意：仅限于学术用途和加速访问github/huggingface，不承诺稳定性保证


In [22]:
CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "nvidia/audio-flamingo-3-hf"

os.environ["HF_HOME"] = CACHE_DIR

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    cache_dir=CACHE_DIR,
)

print("Audio Flamingo 3 model loaded.")

KeyboardInterrupt: 

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [ ]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="flamingo3_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [ ]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "audio", "path": str(wav_path)},
                {"type": "text",  "text": USER_PROMPT},
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=64)
    output = processor.batch_decode(
        generated_ids[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )
    return output[0]

In [ ]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            cleaned = raw.strip().strip("'\".,;:!?").capitalize()
            pred = cleaned if cleaned in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [ ]:
# import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
# from data_split import create_test_csv

# csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
# if not csv.exists() or csv.stat().st_size < 30:
#     create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
# audio_dir = PROJECT_ROOT / "data/raw/Pitt"
# evaluate_dataset(csv, audio_dir, "Pitt-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Lu, exists=True


Lu-raw:   1%|▏         | 1/74 [00:04<05:25,  4.45s/it]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-raw:   3%|▎         | 2/74 [00:07<04:36,  3.83s/it]

  DEBUG [1] session=F22_001 raw="Dementia'" pred=Dementia


Lu-raw:   4%|▍         | 3/74 [00:10<04:03,  3.43s/it]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-raw: 100%|██████████| 74/74 [03:17<00:00,  2.67s/it]

[Lu-raw]
  Accuracy:    0.5135
  F1:          0.6786
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

In [ ]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
# evaluate_dataset(csv, audio_dir, "Lu-Demucs")

[Lu-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Demucs, exists=True


Lu-Demucs:   1%|▏         | 1/74 [00:02<03:18,  2.72s/it]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Demucs:   3%|▎         | 2/74 [00:05<03:35,  2.99s/it]

  DEBUG [1] session=F22_001 raw="Dementia'" pred=Dementia


Lu-Demucs:   4%|▍         | 3/74 [00:08<03:22,  2.86s/it]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Demucs: 100%|██████████| 74/74 [03:10<00:00,  2.58s/it]

[Lu-Demucs]
  Accuracy:    0.5135
  F1:          0.6786
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

In [ ]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
# evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

[Lu-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Denoiser, exists=True


Lu-Denoiser:   1%|▏         | 1/74 [00:02<03:11,  2.63s/it]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Denoiser:   3%|▎         | 2/74 [00:05<03:29,  2.90s/it]

  DEBUG [1] session=F22_001 raw="Dementia'" pred=Dementia


Lu-Denoiser:   4%|▍         | 3/74 [00:08<03:14,  2.74s/it]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Denoiser: 100%|██████████| 74/74 [03:07<00:00,  2.53s/it]

[Lu-Denoiser]
  Accuracy:    0.5135
  F1:          0.6786
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

In [ ]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
# evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

[Lu-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-FRCRN_SE, exists=True


Lu-FRCRN_SE:   1%|▏         | 1/74 [00:02<03:08,  2.58s/it]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   3%|▎         | 2/74 [00:04<02:58,  2.47s/it]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   4%|▍         | 3/74 [00:07<02:57,  2.50s/it]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE: 100%|██████████| 74/74 [03:05<00:00,  2.51s/it]

[Lu-FRCRN_SE]
  Accuracy:    0.5135
  F1:          0.6786
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

In [ ]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
# evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

[Lu-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-MossFormer, exists=True


Lu-MossFormer:   1%|▏         | 1/74 [00:02<03:13,  2.65s/it]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-MossFormer:   3%|▎         | 2/74 [00:05<03:00,  2.51s/it]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-MossFormer:   4%|▍         | 3/74 [00:07<02:59,  2.52s/it]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-MossFormer: 100%|██████████| 74/74 [03:05<00:00,  2.50s/it]

[Lu-MossFormer]
  Accuracy:    0.5135
  F1:          0.6786
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

In [ ]:
# csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
# audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
# evaluate_dataset(csv, audio_dir, "Lu-Resemble")

[Lu-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Resemble, exists=True


Lu-Resemble:   1%|▏         | 1/74 [00:02<03:17,  2.71s/it]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Resemble:   3%|▎         | 2/74 [00:05<03:03,  2.55s/it]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Resemble:   4%|▍         | 3/74 [00:07<03:02,  2.57s/it]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Resemble: 100%|██████████| 74/74 [03:08<00:00,  2.55s/it]

[Lu-Resemble]
  Accuracy:    0.5135
  F1:          0.6786
  Control Acc: 0.0000
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0
